# Block 6 · Neural networks & responsible ML
> Data Mining · SGH Warsaw School of Economics (SMMD-ADA / SMMD-AAB)
> Course anchor dataset: retail-credit / PD portfolio

**Business framing.** The last model family enters the tournament — a multilayer perceptron on the same PD task, against the same baseline. Then the course's most important lab section: **audit our own model for fairness** by sex, run the "just remove sex" experiment, and watch the impossibility theorem show up in our own numbers.

**Learning goals for this lab**
1. Train an MLP in a leakage-safe pipeline; read its loss curve.
2. Feel the learning rate; use early stopping as the leash.
3. Run the final tournament: MLP vs GBDT vs the logistic baseline.
4. Audit a model's fairness (decline rates, FPR, FNR by group) — and see what feature removal does and does not fix.

**How to work.** Run cells top to bottom. Before you trust any result: **Kernel → Restart & Run All.**

In [ ]:
import sys
sys.path.append("../lecture_1")   # shared course data loader

# No warnings.filterwarnings("ignore") here on purpose: if the MLP raises a
# ConvergenceWarning, that IS today's lesson - read it, then adjust the leash.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option('display.max_columns', 30)

from finance_data import load_taiwan, taiwan_features
df = load_taiwan()

X = taiwan_features(df)
y = df["default"]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42)
print(X_train.shape, X_test.shape)

## 1. An MLP, the honest way
Scaler in the pipeline (gradient descent limps on unscaled features), early stopping on (the network's everyday leash), seed fixed (random initialisation!).

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

mlp = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(32, 16), early_stopping=True,
                  random_state=42, max_iter=500))
mlp.fit(X_train, y_train)
auc_mlp = roc_auc_score(y_test, mlp.predict_proba(X_test)[:, 1])
print(f"MLP (32,16) test AUC: {auc_mlp:.3f}")

In [ ]:
# Read the training run: loss down forever, validation peaks then decays.
nn = mlp.named_steps["mlpclassifier"]
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(nn.loss_curve_, color="#2E6DA4", label="training loss")
ax2 = ax.twinx()
ax2.plot(nn.validation_scores_, color="#C83C28", label="validation accuracy")
ax.axvline(np.argmax(nn.validation_scores_), ls=":", color="gray")
ax.set_xlabel("epoch"); ax.set_ylabel("training loss")
ax2.set_ylabel("validation accuracy")
plt.tight_layout(); plt.show()
print(f"stopped after {len(nn.loss_curve_)} epochs "
      f"(best validation at epoch {int(np.argmax(nn.validation_scores_))})")

> **Parameter count check** (slide arithmetic, live): layers 10→32→16→1 give 10·32+32 + 32·16+16 + 16+1 = **897 parameters** — comfortable on 21,000 training rows; the same count needed a much tighter leash on Block 1's 2,800-row world.

In [ ]:
sum(w.size for w in nn.coefs_) + sum(b.size for b in nn.intercepts_)

## 2. Feel the learning rate
Same architecture, SGD solver, three step sizes. Too high overshoots the narrow valley and settles *worse*; too low is asleep.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
for lr, color in [(3.0, "#C83C28"), (0.3, "#2E6DA4"), (0.00001, "#8C959E")]:
    m = make_pipeline(StandardScaler(),
                      MLPClassifier(hidden_layer_sizes=(32, 16), solver="sgd",
                                    learning_rate_init=lr, max_iter=80,
                                    random_state=42))
    m.fit(X_train, y_train)
    ax.plot(m.named_steps["mlpclassifier"].loss_curve_, color=color,
            label=f"lr = {lr}")
ax.set_xlabel("epoch"); ax.set_ylabel("training loss"); ax.legend()
plt.tight_layout(); plt.show()

## 3. The final tournament
Three families, three blocks, one baseline. Same split, same metric.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

models = {
    "logistic (Block 2)": make_pipeline(StandardScaler(),
                                        LogisticRegression(max_iter=2000)),
    "GBDT tuned (Block 3)": HistGradientBoostingClassifier(
        learning_rate=0.03, max_leaf_nodes=7, min_samples_leaf=60,
        early_stopping=True, random_state=42),
    "MLP (Block 6)": mlp,
}
proba = {}
for name, m in models.items():
    m.fit(X_train, y_train)
    proba[name] = m.predict_proba(X_test)[:, 1]
    print(f"{name:22s} test AUC {roc_auc_score(y_test, proba[name]):.3f}")

The final order on the real portfolio: **GBDT (0.773) > MLP (0.759) > logistic (0.731)**. The MLP clears the baseline — complexity earned — but the tabular crown stays with boosted trees, exactly as the benchmark literature predicts. On Block 1's small synthetic world the same ladder collapsed to a tie: the ordering is an empirical question, decided per dataset, and now you own the machinery to decide it honestly.

## 4. The fairness audit
Sex is an unambiguously protected attribute in credit. Our GBDT, at Block 3's cost-optimal threshold (0.26), audited by sex — decline rate, FPR (good clients wrongly declined), FNR (defaulters missed). Note `default` base rates first.

In [ ]:
def group_report(pred, y_true, group_mask, names=("group A", "group B")):
    rows = {}
    for g, name in [(group_mask, names[0]), (~group_mask, names[1])]:
        yt = y_true[g]; pg = pred[g]
        rows[name] = {
            "n": int(g.sum()),
            "base default": yt.mean(),
            "decline rate": pg.mean(),
            "FPR (wrongly declined)": ((pg == 1) & (yt == 0)).sum() / (yt == 0).sum(),
            "FNR (missed defaulters)": ((pg == 0) & (yt == 1)).sum() / (yt == 1).sum(),
        }
    return pd.DataFrame(rows).T.round(3)

female = (df.loc[X_test.index, "sex"] == 2).to_numpy()
y_arr = y_test.to_numpy()

pred_gbdt = proba["GBDT tuned (Block 3)"] >= 0.26
group_report(pred_gbdt, y_arr, female, names=("female", "male"))

Read it slowly: the gaps are **modest and track the base-rate difference** (decline 25.6% vs 22.3% on base rates 23.9% vs 21.0%) — this model does not amplify. It can go the other way: on the synthetic portfolio, an age audit once turned a 3.6 pp base gap into an 18 pp decline gap. You only know after you measure.

### 4a. The "just remove sex" experiment
Fairness through unawareness: drop the protected attribute and refit.

In [ ]:
X2 = X.drop(columns=["female"])
gbdt2 = HistGradientBoostingClassifier(
    learning_rate=0.03, max_leaf_nodes=7, min_samples_leaf=60,
    early_stopping=True, random_state=42)
gbdt2.fit(X2.loc[X_train.index], y_train)
proba2 = gbdt2.predict_proba(X2.loc[X_test.index])[:, 1]
print(f"AUC without sex: {roc_auc_score(y_test, proba2):.3f}")
group_report(proba2 >= 0.26, y_arr, female, names=("female", "male"))

Two lessons, both honest:
1. **Removal changes essentially nothing** — every number is identical with or without the column. The model never needed `sex`: six months of payment behaviour already carries whatever group structure exists. That is **proxy discrimination** stated at its purest — deleting the label does not delete the information.
2. Corollary: "fairness through unawareness" here is free and toothless at once. If the measured gaps needed fixing, the fix would have to act on **decisions** (thresholds, constraints) — a governance choice, not a `fit()` argument. (The impossibility theorem still lurks: with different base rates you cannot equalise decline rates, error rates and calibration simultaneously.)

## 5. Exercises
1. **Width sweep:** hidden layers (4,), (32,16), (256,128) under 5-fold CV AUC. Where is the U-curve — and does any width beat the baseline?
2. **Leash off:** refit the MLP with `early_stopping=False, max_iter=2000`. Compare train vs test AUC. Name what you observe using course vocabulary.
3. **Smoke test:** can your MLP perfectly memorise 100 training rows? (It should — if not, debug the pipeline, not the patience.)
4. **Calibration:** draw the reliability diagram for the MLP (Block 3 code reuses verbatim). Would you price loans with these probabilities?
5. **Fairness × threshold:** recompute the group report at thresholds 0.5 and 0.26. Does the fairness picture depend on the threshold? And repeat the audit **by age group** (`age < 35`) — which gaps appear?
6. **Your project:** run `group_report` on your own project's model with any sensible group split. Put the table and two honest sentences in your limitations section — defence juries love it, and so do regulators.

*Hand-in for the project team: your final tournament table and a fairness table for your model, each with a two-sentence honest reading.*

**"Done" for today** = your MLP verdict matches your tournament table, and your fairness audit has numbers in it — this block's sentence for the defence: measured, not assumed.

In [ ]:
# scratch cell for the exercises